# 🔄 The Pivot — HF TRL GRPOTrainer
**Meta PyTorch OpenEnv Hackathon 2026**

This notebook trains the startup pivot advisor using **Hugging Face TRL's official `GRPOTrainer`**.

### How it works
- **300 gradient steps total** — 60 steps per curriculum tier × 5 tiers
- **Curriculum**: Tier 1 (easy B2C) → Tier 5 (complex FinTech), advancing in difficulty
- **Reward function**: wraps the OpenEnv pivot environment — TRL asks the model for an action, the env returns a reward
- **Model**: Qwen2.5-1.5B-Instruct + QLoRA (4-bit, fits T4)

### Steps
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells top to bottom (~60–90 min)

In [ ]:
# Cell 1 — Check GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — go to Runtime → Change runtime type → T4')
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Please enable GPU before continuing!'

In [ ]:
# Cell 2 — Install packages
# trl>=0.12.0 includes GRPOTrainer — the official GRPO implementation from HuggingFace
%%capture
!pip install -q openenv-core \
    "transformers>=4.45.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.34.0" \
    "trl>=0.12.0" \
    "datasets>=2.20.0" \
    wandb pydantic numpy python-dotenv
print('done')

In [ ]:
# Cell 3 — Verify TRL version
import trl, transformers, peft
print(f'TRL version       : {trl.__version__}')
print(f'Transformers      : {transformers.__version__}')
print(f'PEFT              : {peft.__version__}')

from trl import GRPOTrainer, GRPOConfig
print('\n✅ GRPOTrainer imported successfully')

In [ ]:
# Cell 4 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_trl'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Saving to: {SAVE_DIR}')

In [ ]:
# Cell 5 — Clone project + imports
import os, sys
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler
sys.path.insert(0, '/content/meta_scaler')
os.chdir('/content/meta_scaler')

try:
    from models import CoFounderAction, ActionType, CoFounderObservation
    from server.cofounder_environment import CoFounderEnvironment
    from server.prompt_encoder import encode_to_messages
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports OK!')
    print('Actions available:', len(list(ActionType)))
    print('Difficulty ladder:', DIFFICULTY_LADDER)
except ImportError as e:
    print(f'❌ Import error: {e}')

In [ ]:
# Cell 6 — W&B login
import wandb
wandb.login()   # paste your key from wandb.ai/settings
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

In [ ]:
# Cell 7 — Load model (Qwen2.5-1.5B + QLoRA, same config as train_colab.ipynb)
# 4-bit nf4 quantization → ~2 GB VRAM for weights, fits T4 (15 GB).
# Only LoRA adapters (r=8 on q_proj + v_proj) are trainable — ~3M params.
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME     = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE         = 'cuda'
MAX_SEQ_LEN      = 512
MAX_NEW_TOKENS   = 64    # 64 tokens needed for full DECISION: format + 12-action reasoning
DEMO_MAX_TOKENS  = 200   # used in demo cell for full reasoning output

print(f'Loading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map='auto',
    trust_remote_code=True,
)

model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
))
model.enable_input_require_grads()
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model ready!  GPU: {used:.2f} GB used / {total:.1f} GB total  ({total-used:.1f} GB free)')

In [ ]:
# Cell 8 — Helpers
import re, random

# Full 12-action map — includes all CoFounder actions added in plan.md Step 6
ACTION_MAP = {
    'execute':            ActionType.EXECUTE,
    'pivot':              ActionType.PIVOT,
    'research':           ActionType.RESEARCH,
    'fundraise':          ActionType.FUNDRAISE,
    'hire':               ActionType.HIRE,
    'cut_costs':          ActionType.CUT_COSTS,
    'cut':                ActionType.CUT_COSTS,
    'sell':               ActionType.SELL,
    'launch_feature':     ActionType.LAUNCH_FEATURE,
    'launch':             ActionType.LAUNCH_FEATURE,
    'marketing_campaign': ActionType.MARKETING_CAMPAIGN,
    'marketing':          ActionType.MARKETING_CAMPAIGN,
    'set_pricing':        ActionType.SET_PRICING,
    'pricing':            ActionType.SET_PRICING,
    'fire':               ActionType.FIRE,
    'partnership':        ActionType.PARTNERSHIP,
}


def _parse(text: str) -> ActionType:
    """
    Parse action from model output.
    Handles two formats:
      1. Structured: looks for 'DECISION: ACTION_WORD' line (primary format)
      2. Simple fallback: first word of output
    """
    for line in text.split('\n'):
        stripped = line.strip().lower()
        for prefix in ('decision:', 'recommendation:', 'action:'):
            if stripped.startswith(prefix):
                rest = stripped[len(prefix):].strip()
                word = re.sub(r'[^a-z_]', '', rest.split()[0]) if rest.split() else ''
                if word in ACTION_MAP:
                    return ACTION_MAP[word]
    # Fallback: first word
    w = re.sub(r'[^a-z_]', '', text.lower().split()[0]) if text.strip() else 'execute'
    return ACTION_MAP.get(w, ActionType.EXECUTE)


def heuristic_action(obs) -> ActionType:
    """
    Rule-based policy used for the 9 continuation steps in the 10-step
    lookahead reward. Updated for 12 actions and new observation fields.
    NOT the model — just fast rules to fill the lookahead steps.
    """
    # Emergency: runway critical
    if obs.runway_remaining <= 2:
        return ActionType.SELL
    if obs.runway_remaining < 4:
        return ActionType.CUT_COSTS

    # Product health issues
    pmf = getattr(obs, 'pmf_score', 0.5)
    if pmf < 0.35:
        return ActionType.LAUNCH_FEATURE   # product needs work first

    # Team burnout — stop adding cost before morale recovers
    morale = getattr(obs, 'team_morale', 0.7)
    if morale < 0.40:
        if obs.runway_remaining > 6:
            return ActionType.FIRE         # triage headcount to stop burn
        return ActionType.CUT_COSTS

    # Unit economics broken — fix before spending on marketing
    ltv_cac = getattr(obs, 'ltv_cac_ratio', 3.0)
    if ltv_cac < 1.5 and pmf >= 0.50:
        return ActionType.MARKETING_CAMPAIGN   # product ready, optimize CAC

    # High churn → investigate
    if getattr(obs, 'churn_rate', 0) > 0.15:
        return ActionType.RESEARCH

    # Good NPS + headroom → hire to accelerate
    if getattr(obs, 'nps_score', 50) > 65 and obs.runway_remaining > 10:
        return ActionType.HIRE

    return ActionType.EXECUTE


def evaluate_action_reward(scenario: dict, seed: int, action: ActionType,
                           lookahead: int = 10, sector: str = 'b2b_enterprise') -> float:
    """
    Reward signal for TRL reward function.
    Takes the model's chosen action for step 1, then uses a fast
    heuristic policy for the next (lookahead-1) steps.
    Returns total reward over `lookahead` steps.

    Why 10-step lookahead instead of 1-step?
    - Single-step reward is too noisy (random events dominate)
    - Full episode (60 steps) would be slow and hard to attribute to step-1 action
    - 10 steps captures immediate consequences of the chosen action
    """
    env  = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
    obs  = env.reset()
    total = 0.0
    for i in range(lookahead):
        act = action if i == 0 else heuristic_action(obs)
        obs = env.step(CoFounderAction(action_type=act))
        total += float(obs.reward or 0.0)
        if obs.done:
            break
    return total


print('✅ Helpers ready — ACTION_MAP has', len(ACTION_MAP), 'entries (12 unique actions)')

In [ ]:
# Cell 9 — Reward function + dataset builder
# ─────────────────────────────────────────────────────────────────────────────
# TRL's GRPOTrainer needs:
#   1. A Dataset with a 'prompt' column (the text the model sees)
#   2. A reward_fn(completions, **dataset_columns) → List[float]
#
# Extra dataset columns (env_key) are passed as kwargs to reward_fn.
# This is how we reconnect each model completion back to its env state.
# ─────────────────────────────────────────────────────────────────────────────
from datasets import Dataset

# Global registry: env_key → {scenario, seed}
# Populated by build_tier_dataset(), read by pivot_reward_fn()
_ENV_REGISTRY = {}


def build_tier_dataset(scenario_name: str, n_seeds: int = 30) -> Dataset:
    """
    Build a HuggingFace Dataset for one curriculum tier.
    Each row = initial observation with real sector benchmarks embedded in the prompt.
    """
    from training.market_data import infer_sector_from_scenario
    sector     = infer_sector_from_scenario(scenario_name)
    curriculum = AdaptiveCurriculum(seed=0)
    scenario   = curriculum._all_scenarios.get(scenario_name)
    assert scenario, f'Scenario "{scenario_name}" not found'

    rows = []
    for seed in range(n_seeds):
        env = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
        obs = env.reset()
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs, sector=sector), tokenize=False, add_generation_prompt=True)
        key = f"{scenario_name}_s{seed}"
        _ENV_REGISTRY[key] = {'scenario': scenario, 'seed': seed, 'sector': sector}
        rows.append({'prompt': prompt, 'env_key': key})

    return Dataset.from_list(rows)


def pivot_reward_fn(completions, env_key, **kwargs) -> list:
    """
    TRL reward function — called by GRPOTrainer after model generates completions.
    Parses action from DECISION: line (supports all 12 actions), runs 10-step
    lookahead in the real CoFounderEnvironment, returns total reward.
    """
    rewards = []
    for completion, key in zip(completions, env_key):
        try:
            data   = _ENV_REGISTRY[key]
            action = _parse(completion)   # handles DECISION: format, all 12 actions
            r      = evaluate_action_reward(
                scenario  = data['scenario'],
                seed      = data['seed'],
                action    = action,
                lookahead = 10,
                sector    = data.get('sector', 'b2b_enterprise'),
            )
            rewards.append(r)
        except Exception:
            rewards.append(0.0)
    return rewards


# Quick sanity check
_curriculum_check = AdaptiveCurriculum(seed=0)
_sc   = _curriculum_check._all_scenarios[DIFFICULTY_LADDER[0]]
_test = evaluate_action_reward(_sc, seed=0, action=ActionType.EXECUTE)
print(f'Reward function sanity check: EXECUTE on {DIFFICULTY_LADDER[0]} seed 0 → reward = {_test:.1f}')
print('✅ Dataset builder + reward function ready')

In [ ]:
# Cell 10 — 5-Tier Curriculum Training with TRL GRPOTrainer
# ─────────────────────────────────────────────────────────────────────────────
# Architecture:
#   • 5 sequential GRPOTrainer runs, one per curriculum tier
#   • 60 gradient steps per tier × 5 tiers = 300 total steps
#   • Same model object is passed to each trainer — weights carry over
#   • Dataset rebuilt fresh for each tier (harder scenarios as tiers advance)
#
# GRPO mechanics (TRL handles this internally):
#   1. For each prompt, generate num_generations=4 different completions
#   2. Call pivot_reward_fn() to score each completion
#   3. Compute within-group advantage = (reward - group_mean) / group_std
#   4. Update model: gradient = -log_prob × advantage + KL_penalty
#
# 300 steps × 4 accum × 4 generations = 4,800 env interactions total.
# ─────────────────────────────────────────────────────────────────────────────
import gc, torch

# Training hyperparameters (same family as custom GRPO notebook)
STEPS_PER_TIER  = 60    # gradient steps per difficulty tier
N_SEEDS         = 30    # prompts per tier dataset (cycles ~8× per 60 steps)
LR              = 2e-5  # slightly lower than custom GRPO for TRL stability
NUM_GENERATIONS = 4     # GRPO group size — 4 completions per prompt per step
GRAD_ACCUM      = 4     # accumulate 4 steps before update

# W&B run
run = wandb.init(
    project = WANDB_PROJECT,
    name    = 'trl-grpo-qwen1.5b-cofounder',
    config  = {
        'model':           MODEL_NAME,
        'method':          'TRL GRPOTrainer',
        'env':             'CoFounderEnvironment',
        'n_actions':       12,
        'total_steps':     STEPS_PER_TIER * len(DIFFICULTY_LADDER),
        'steps_per_tier':  STEPS_PER_TIER,
        'num_generations': NUM_GENERATIONS,
        'lr':              LR,
        'lookahead':       10,
        'n_seeds':         N_SEEDS,
        'max_new_tokens':  MAX_NEW_TOKENS,
    },
    tags=['trl','grpo','qwen1.5b','cofounder','hackathon','curriculum'],
)
print(f'W&B: {run.url}\n')

# ── Main training loop — one GRPOTrainer per curriculum tier ──────────────────
tier_results = []   # track mean reward per tier for logging

for tier_idx, scenario_name in enumerate(DIFFICULTY_LADDER):
    print('='*70)
    print(f'🎓  TIER {tier_idx+1}/5 — Scenario: {scenario_name}')
    print(f'    Steps {tier_idx*STEPS_PER_TIER+1}–{(tier_idx+1)*STEPS_PER_TIER} of {STEPS_PER_TIER*len(DIFFICULTY_LADDER)}')
    print('='*70)

    # ── Build dataset for this tier ───────────────────────────────────────────
    tier_dataset = build_tier_dataset(scenario_name, n_seeds=N_SEEDS)
    print(f'Dataset: {len(tier_dataset)} prompts for "{scenario_name}"')

    # ── Configure TRL GRPOTrainer ──────────────────────────────────────────────
    trl_config = GRPOConfig(
        output_dir                  = f'{SAVE_DIR}/tier_{tier_idx+1}',

        # Optimizer
        learning_rate               = LR,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = GRAD_ACCUM,

        # GRPO-specific
        num_generations             = NUM_GENERATIONS,  # completions per prompt
        max_completion_length       = MAX_NEW_TOKENS,
        max_prompt_length           = MAX_SEQ_LEN,
        temperature                 = 0.9,

        # Training schedule
        max_steps                   = STEPS_PER_TIER,
        num_train_epochs            = 999,   # effectively unlimited, max_steps takes over

        # Logging + saving
        logging_steps               = 10,
        save_steps                  = STEPS_PER_TIER,  # save at end of each tier
        report_to                   = 'wandb',

        # CRITICAL: keep 'env_key' column so reward_fn receives it
        remove_unused_columns       = False,

        # Precision
        bf16                        = True,
        dataloader_num_workers      = 0,
    )

    # ── Create and run trainer ────────────────────────────────────────────────
    try:
        trainer = GRPOTrainer(
            model            = model,
            reward_funcs     = pivot_reward_fn,
            args             = trl_config,
            train_dataset    = tier_dataset,
            processing_class = tokenizer,   # TRL 0.12+ parameter name
        )
    except TypeError:
        # Fallback for TRL versions that use 'tokenizer=' instead of 'processing_class='
        trainer = GRPOTrainer(
            model         = model,
            reward_funcs  = pivot_reward_fn,
            args          = trl_config,
            train_dataset = tier_dataset,
            tokenizer     = tokenizer,
        )

    trainer.train()

    # ── Log tier results ──────────────────────────────────────────────────────
    # Quick eval: run 5 test episodes on this scenario to measure improvement
    model.eval()
    test_rewards = []
    for test_seed in range(40, 45):  # seeds not seen during training
        env = CoFounderEnvironment(
            scenario=_ENV_REGISTRY[f'{scenario_name}_s0']['scenario'],
            rng_seed=test_seed)
        obs = env.reset()
        ep_r = 0.0
        for _ in range(60):
            prompt = tokenizer.apply_chat_template(
                encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
            inp = tokenizer(prompt, return_tensors='pt',
                            truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                     do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
            text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
            action = _parse(text)
            obs = env.step(CoFounderAction(action_type=action))
            ep_r += obs.reward or 0.0
            del inp, out
            torch.cuda.empty_cache()
            if obs.done:
                break
        test_rewards.append(ep_r)

    mean_r   = sum(test_rewards) / len(test_rewards)
    survived = sum(1 for r in test_rewards if r > 0) / len(test_rewards)
    tier_results.append({'tier': tier_idx+1, 'scenario': scenario_name,
                         'mean_reward': mean_r, 'survival_rate': survived})

    print(f'\n📊 Tier {tier_idx+1} eval: mean_reward={mean_r:+.1f}  survival={survived:.0%}')
    wandb.log({
        'tier':             tier_idx + 1,
        'tier_mean_reward': mean_r,
        'tier_survival':    survived,
        'scenario':         scenario_name,
        'global_step':      (tier_idx + 1) * STEPS_PER_TIER,
    })

    # Free trainer memory before next tier
    del trainer, tier_dataset
    gc.collect()
    torch.cuda.empty_cache()
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('✅ ALL 5 TIERS COMPLETE — 300 gradient steps via TRL GRPOTrainer')
print('='*70)
print(f'{"Tier":>4} | {"Scenario":>20} | {"Mean Reward":>12} | {"Survival":>8}')
print('-'*55)
for t in tier_results:
    print(f'{t["tier"]:>4} | {t["scenario"]:>20} | {t["mean_reward"]:>+12.1f} | {t["survival_rate"]:>7.0%}')

In [ ]:
# Cell 11 — Save final model + close W&B
FINAL = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
print(f'✅ TRL-trained model saved → {FINAL}')

import json
with open(f'{SAVE_DIR}/tier_results.json', 'w') as f:
    json.dump(tier_results, f, indent=2)
print(f'✅ Tier results saved → {SAVE_DIR}/tier_results.json')

wandb.finish()
print('✅ W&B closed')

In [ ]:
# Cell 12 — Push TRL-trained LoRA to HF Hub
# ─────────────────────────────────────────────────────────────────────────────
# After running:
#   1. Model is at huggingface.co/Harshit-Makraria/the-pivot-lora-trl
#   2. This is your TRL-trained model (separate from the custom GRPO one)
# ─────────────────────────────────────────────────────────────────────────────
from huggingface_hub import HfApi

HF_USERNAME  = 'Harshit-Makraria'
HF_REPO_NAME = 'the-pivot-lora-trl'
HF_MODEL_ID  = f'{HF_USERNAME}/{HF_REPO_NAME}'

HF_TOKEN = input('Paste your HF token (hf_...): ').strip()

print(f'Pushing to {HF_MODEL_ID} ...')
model.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)

card = f"""---
base_model: Qwen/Qwen2.5-1.5B-Instruct
tags:
  - peft
  - lora
  - rl
  - grpo
  - trl
  - startup
  - hackathon
license: mit
---

# The CoFounder Strategist — TRL GRPOTrainer Model

Fine-tuned with **HuggingFace TRL `GRPOTrainer`** on the [CoFounder Strategist](https://github.com/Harshit-Makraria/meta_scaler) OpenEnv environment.

## What it does
Acts as an AI co-founder advisor across **5 startup scenarios** (B2C SaaS, B2B Enterprise,
Marketplace, FinTech, Consumer App). Chooses from **12 strategic actions** each month to
maximize a 30+ dimensional balanced scorecard spanning PMF, team morale, unit economics,
and founder trust.

## Training
- **300 gradient steps** (60 per tier × 5 curriculum tiers)
- **10-step lookahead reward** from live CoFounderEnvironment
- **GRPO group size**: 4 completions per prompt
- **Balanced scorecard** reward: survival (30%) + PMF (20%) + morale (20%) + LTV:CAC (15%) + trust (15%)

## Actions supported
`EXECUTE` · `PIVOT` · `RESEARCH` · `FUNDRAISE` · `HIRE` · `CUT_COSTS` · `SELL` ·
`LAUNCH_FEATURE` · `MARKETING_CAMPAIGN` · `SET_PRICING` · `FIRE` · `PARTNERSHIP`

## Output format
```
DECISION: launch_feature
```

Built for the **Meta PyTorch OpenEnv Hackathon 2026**.
"""

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo='README.md',
    repo_id=HF_MODEL_ID,
    repo_type='model',
)
print(f'\n✅ TRL model live at: https://huggingface.co/{HF_MODEL_ID}')

In [ ]:
# Cell 13 — Demo: watch the TRL-trained model make decisions
# Shows the model choosing from 12 actions across all 5 scenarios after TRL training
model.eval()

print('\n🎮  POST-TRAINING DEMO — TRL GRPOTrainer Model (CoFounderEnvironment)')
print('=' * 70)
print(f'{"Scenario":22s} | {"Outcome":14s} | {"Reward":>8} | {"Steps":>5} | {"Pivots":>6} | Key actions')
print('-' * 70)

for scenario_name in DIFFICULTY_LADDER:
    curriculum = AdaptiveCurriculum(seed=0)
    scenario   = curriculum._all_scenarios[scenario_name]
    env        = CoFounderEnvironment(scenario=scenario, rng_seed=999)
    obs        = env.reset()

    total_r      = 0.0
    steps        = 0
    pivots       = 0
    action_tally = {}

    for _ in range(60):
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        text   = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        action = _parse(text)
        obs    = env.step(CoFounderAction(action_type=action))
        total_r += obs.reward or 0.0
        steps   += 1
        if action == ActionType.PIVOT:
            pivots += 1
        action_tally[action.value] = action_tally.get(action.value, 0) + 1
        del inp, out
        torch.cuda.empty_cache()
        if obs.done:
            break

    outcome = '✅ SURVIVED' if obs.runway_remaining > 0 else '❌ BANKRUPT'
    top2    = sorted(action_tally.items(), key=lambda x: -x[1])[:2]
    top_str = ', '.join(f'{a}×{n}' for a, n in top2)
    print(f'{scenario_name:22s} | {outcome:14s} | {total_r:>+8.1f} | {steps:>5} | {pivots:>6} | {top_str}')

print('='*70)
print('\nWhat good TRL training looks like:')
print('  • Survived easy scenarios (b2c_saas, b2b_enterprise)')
print('  • 1–3 PIVOTs on hard scenarios — not 0 (stubborn) or 10+ (panic)')
print('  • Mix of actions: LAUNCH_FEATURE, RESEARCH, HIRE — not just EXECUTE')
print('  • Hard scenario (consumer_app, fintech) reward lower than easy — healthy gradient')